# MMLU Fine-tuned Models

### BERT 

In [43]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import AutoTokenizer
from transformers import AutoModel, AutoModelForMultipleChoice

pd.options.plotting.backend = "plotly"

In [22]:
df = pd.read_csv("../data/auxiliary_train/arc_easy.csv", header = None)
df.columns = ["Question", "A", "B", "C", "D", "answer"]
df.head()

,Question,A,B,C,D,answer
0,Which factor will most likely cause a person t...,a leg muscle relaxing after exercise,a bacterial population in the bloodstream,several viral particles on the skin,carbohydrates being digested in the stomach,B
1,Lichens are symbiotic organisms made of green ...,carbon dioxide,food,protection,water,B
2,When a switch is used in an electrical circuit...,cause the charge to build.,increase and decrease the voltage.,cause the current to change direction.,stop and start the flow of current.,D
3,Which of the following is an example of an ass...,contact lens,motorcycle,raincoat,coffee pot,A
4,"Rocks are classified as igneous, metamorphic, ...",their color,their shape,how they formed,the minerals they contain,C


In [23]:
df["text"] = (
            "Q: " + df["Question"].astype(str) + 
            "A) " + df["A"].astype(str) +
            "B) " + df["B"].astype(str) +
            "C) " + df["C"].astype(str) + 
            "D) " + df["D"].astype(str) 
            )

print(df["text"])

letter2id = {"A":0, "B":1, "C":2, "D":3}

df["label"] = df["answer"].map(letter2id)
df.head()

0       Q: Which factor will most likely cause a perso...
1       Q: Lichens are symbiotic organisms made of gre...
2       Q: When a switch is used in an electrical circ...
3       Q: Which of the following is an example of an ...
4       Q: Rocks are classified as igneous, metamorphi...
                              ...                        
2237    Q: Iron oxides, such as rust, form when iron m...
2238    Q: When water evaporates from Earth's surface ...
2239    Q: Which process directly adds carbon into the...
2240    Q: Scientists think that dolphins and whales m...
2241    Q: A particular organism is able to survive in...
Name: text, Length: 2242, dtype: object


,Question,A,B,C,D,answer,text,label
0,Which factor will most likely cause a person t...,a leg muscle relaxing after exercise,a bacterial population in the bloodstream,several viral particles on the skin,carbohydrates being digested in the stomach,B,Q: Which factor will most likely cause a perso...,1
1,Lichens are symbiotic organisms made of green ...,carbon dioxide,food,protection,water,B,Q: Lichens are symbiotic organisms made of gre...,1
2,When a switch is used in an electrical circuit...,cause the charge to build.,increase and decrease the voltage.,cause the current to change direction.,stop and start the flow of current.,D,Q: When a switch is used in an electrical circ...,3
3,Which of the following is an example of an ass...,contact lens,motorcycle,raincoat,coffee pot,A,Q: Which of the following is an example of an ...,0
4,"Rocks are classified as igneous, metamorphic, ...",their color,their shape,how they formed,the minerals they contain,C,"Q: Rocks are classified as igneous, metamorphi...",2


In [24]:
X_train, X_temp, y_train, y_temp = train_test_split(df["text"], df["label"], test_size = 0.3, random_state=2018, stratify=df["label"])

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=2018, stratify=y_temp
)

In [30]:
bert = AutoModel.from_pretrained('bert-base-uncased')
bert_fine_tuned = AutoModel.from_pretrained('bert-base-uncased')

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

In [26]:
seq_len = [len(i.split()) for i in X_train]

pd.Series(seq_len).hist(bins=30)

### Encodings

In [35]:
ppairs = []
for _, row in df.iterrows():
    pairs = [
        (row["Question"], row["A"]),
        (row["Question"], row["B"]),
        (row["Question"], row["C"]),
        (row["Question"], row["D"]),
    ]
    ppairs.append(pairs)

all_q = [q for pairs in ppairs for (q,_) in pairs]
all_o = [o for pairs in ppairs for (_,o) in pairs]

enc = tokenizer(all_q, all_o, padding = True, truncation = True, max_length = 256, return_tensors = "pt")



In [ ]:
from torch.utils.data import Dataset, DataLoader

letters = ["A", "B", "C", "D"]
label_map = {"A": 0 , "B":1, "C":2, "D":3}

class MCDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.df["label"] = self.df["answer"].map(label_map)
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        choices = [row[c] for c in letters]
        return {
            "question": row["Question"],
            "choices": choices,
            "label": row["label"]
        }

def mc_collate(batch):
    all_q = [ex["question"] for ex in batch for _ in range(4)]
    all_opt = [opt for ex in batch for opt in ex["choices"]]

    enc = tokenizer(all_q, all_opt, padding = True, truncation = True, max_length = 256, return_tensors="pt")

    B = len(batch); C=4; L=enc["input_ids"].size(1)

    collated = {
        "input_ids": enc["input_ids"].view(B,C,L),
        "attention_masks": enc["attention_mask"].view(B,C,L),
        "labels": torch.tensor([ex["label"] for ex in batch], dtype=torch.long),
    }

    if "token_type_ids" in enc:
        collated["token_type_ids"] = enc["token_type_ids"].view(B,C,L)

    return collated

### Shaping for Tensors

In [ ]:
B = len(ppairs)
C = 4
L = enc["input_ids"].size(1)

torch.Size([8968, 122])


In [42]:
input_ids = enc["input_ids"].view(B,C,L)
attention_masks = enc["attention_mask"].view(B,C,L)
token_type_ids = enc.get("token_type_ids")
if token_type_ids is not None:
    token_type_id = enc.get("token_type_ids").view(B,C,L)

labels = torch.tensor(df["label"].tolist(), dtype = torch.long)

In [44]:
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

Some weights of BertForMultipleChoice were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
